# 01 — Pré-processamento dos dados

Este notebook transforma os dados brutos em uma base pronta para análise. As transformações são deliberadamente conservadoras: dados ausentes não recebem valores presumidos.

## Objetivos

- Validar a estrutura do CSV original;
- Converter datas e idade para os tipos corretos;
- Tornar ausências categóricas explícitas como `Desconhecido`;
- Criar variáveis analíticas, como ano do óbito, faixa etária e dias até o óbito;
- Salvar uma versão processada em `data/processed/`.

In [1]:
from pathlib import Path
import sys

RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / 'src').exists():
    RAIZ_PROJETO = RAIZ_PROJETO.parent
sys.path.insert(0, str(RAIZ_PROJETO / 'src'))

from data_processing import (
    carregar_dados,
    preparar_dados,
    relatorio_qualidade,
    salvar_dados_processados,
)

## Carregamento e validação

In [2]:
dados_brutos = carregar_dados()
print(f'Registros: {len(dados_brutos):,}')
print(f'Colunas: {dados_brutos.shape[1]}')
dados_brutos.head()

Registros: 11,124
Colunas: 16


,name,date_of_event,age,citizenship,event_location,event_location_district,event_location_region,date_of_death,gender,took_part_in_the_hostilities,place_of_residence,place_of_residence_district,type_of_injury,ammunition,killed_by,notes
0,'Abd a-Rahman Suleiman Muhammad Abu Daghash,2023-09-24,32.0,Palestinian,Nur Shams R.C.,Tulkarm,West Bank,2023-09-24,M,NaN,Nur Shams R.C.,Tulkarm,gunfire,live ammunition,Israeli security forces,Fatally shot by Israeli forces while standing ...
1,Usayed Farhan Muhammad 'Ali Abu 'Ali,2023-09-24,21.0,Palestinian,Nur Shams R.C.,Tulkarm,West Bank,2023-09-24,M,NaN,Nur Shams R.C.,Tulkarm,gunfire,live ammunition,Israeli security forces,Fatally shot by Israeli forces while trying to...
2,'Abdallah 'Imad Sa'ed Abu Hassan,2023-09-22,16.0,Palestinian,Kfar Dan,Jenin,West Bank,2023-09-22,M,NaN,al-Yamun,Jenin,gunfire,live ammunition,Israeli security forces,Fatally shot by soldiers while firing at them ...
3,Durgham Muhammad Yihya al-Akhras,2023-09-20,19.0,Palestinian,'Aqbat Jaber R.C.,Jericho,West Bank,2023-09-20,M,NaN,'Aqbat Jaber R.C.,Jericho,gunfire,live ammunition,Israeli security forces,Shot in the head by Israeli forces while throw...
4,Raafat 'Omar Ahmad Khamaisah,2023-09-19,15.0,Palestinian,Jenin R.C.,Jenin,West Bank,2023-09-19,M,NaN,Jenin,Jenin,gunfire,live ammunition,Israeli security forces,Wounded by soldiers’ gunfire after running awa...


## Transformações

As datas são convertidas para `datetime`; idade é convertida para valor numérico. Campos categóricos ausentes recebem o rótulo `Desconhecido`, sem inferir características das vítimas.

In [3]:
dados = preparar_dados(dados_brutos)
dados[['date_of_event', 'date_of_death', 'age', 'ano_evento', 'ano_obito', 'faixa_etaria', 'dias_ate_obito']].head()

,date_of_event,date_of_death,age,ano_evento,ano_obito,faixa_etaria,dias_ate_obito
0,2023-09-24,2023-09-24,32.0,2023,2023,25–34,0
1,2023-09-24,2023-09-24,21.0,2023,2023,15–24,0
2,2023-09-22,2023-09-22,16.0,2023,2023,15–24,0
3,2023-09-20,2023-09-20,19.0,2023,2023,15–24,0
4,2023-09-19,2023-09-19,15.0,2023,2023,15–24,0


## Qualidade dos dados

Os valores ausentes de idade permanecem visíveis e devem ser excluídos apenas das análises específicas de idade.

In [4]:
qualidade = relatorio_qualidade(dados)
qualidade

,coluna,tipo,valores_ausentes,percentual_ausente
15,notes,object,280,2.52
2,age,float64,129,1.16
18,faixa_etaria,category,129,1.16
0,name,object,0,0.00
3,citizenship,object,0,0.00
1,date_of_event,datetime64[ns],0,0.00
4,event_location,object,0,0.00
5,event_location_district,object,0,0.00
8,gender,object,0,0.00
9,took_part_in_the_hostilities,object,0,0.00


## Salvamento

A base processada é a entrada do notebook de análise. O arquivo bruto em `data/raw/` não é alterado.

In [5]:
caminho_processado = salvar_dados_processados(dados)
print(f'Dados processados salvos em: {caminho_processado.relative_to(RAIZ_PROJETO)}')

Dados processados salvos em: data/processed/fatalities_preparadas.csv
